# **IX) Variational Mode Decomposition (VMD) - Granger Causality**


1. [**Import Data**](#1-import--data)
2. [**Model Parameters**](#2-parameters)
3. [**Decomposition & Alignment**](#3-alignment)
4. [**Cross-Frequency Granger Testing**](#4-granger)
5. [**Granger on Original Series**](#5-granger--orig)
6. [**LaTeX Output**](#6-latex)


### **1) Import Data** <a id="1-import--data"></a>

In [1]:
import numpy as np
import pandas as pd
from vmdpy import VMD
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import grangercausalitytests
from scipy.stats import f

df = pd.read_csv("data/All_Data_Weekly_transformed.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date").sort_index()

target = "btc_ret"

stationary_predictors = [
    "mp_exp_d", "news_sent", "policy_risk_gr",
    "sp500_ret", "brent_gr", "gold_ret", "hy_ret",
    "gpr_gr", "vix_gr", "dxy_gr", "emv",
    "claims_dlog", "eer_dlog", "ffr_d2", "infl5y_d",
    "gt_infl_d", "gt_recess", "gt_climate"
]

btc = df[target]

### **2) Model Parameters** <a id="2-parameters"></a>

| Parameter | Function | Impact |
| :--- | :--- | :--- |
| **alpha** | Bandwidth | Higher = Sharper, narrower frequency modes; Lower = Wider, overlapping modes. |
| **K** | Mode Count | Total number of components. 3 typically captures: Noise, Cycle, and Trend. |
| **tol** | Tolerance | Convergence threshold. 1e-6 ensures high mathematical precision. |
| **tau** | Time-step | Update rate of the dual ascent. 0 is standard for data fidelity. |
| **DC** | DC Offset | If 1, the first mode captures the mean; 0 is preferred for return series. |
| **init** | Init. Type | 1 = Uniformly spaced frequency initialization (most stable). |
| **MAX_GC_LAG** | Granger Lag | Number of weeks to look back. 1 tests for immediate weekly influence. |

An alpha of 1000 and K=3 are standard for financial data

In [2]:
# VMD parameters: alpha (bandwidth), K (number of modes), tol (tolerance)
alpha, tau, K, DC, init, tol = 1000, 0, 3, 0, 1, 1e-6
MAX_GC_LAG = 6

### **3) Decomposition & Alignment** <a id="3-alignment"></a>

ensures that the IMFs are perfectly aligned with the original time-series index.

In [3]:
# Helper: VMD -> DataFrame aligned to index (fixes length mismatch)
def vmd_to_df(series: pd.Series) -> pd.DataFrame:
    """
    Runs VMD and returns (N, K) DataFrame of modes aligned to the series index.
    Fixes vmdpy cases where output length differs from input length.
    """
    s = series.dropna().astype(float)
    if len(s) <= (MAX_GC_LAG + 5):
        return pd.DataFrame(index=s.index)

    u, _, _ = VMD(s.values, alpha, tau, K, DC, init, tol)  # u: (K, N_out)
    uT = u.T  # (N_out, K)

    n_in = len(s)
    n_out = uT.shape[0]
    n = min(n_in, n_out)

    # Align using last n observations (most defensible for time series)
    idx = s.index[-n:]
    uT = uT[-n:, :]

    imfs = pd.DataFrame(uT, index=idx, columns=[f"IMF{i+1}" for i in range(uT.shape[1])])
    return imfs

### **4) Cross-Frequency Granger Testing** <a id="4-granger"></a>

- Function: VMD + Granger causality
- Direction: VAR_IMF(j) -> BTC_IMF(i)

In [4]:

results = []# To store results

def vmd_granger(x_btc: pd.Series, y_var: pd.Series, max_gc_lag: int = 1):
    """
    Decompose BTC and VAR with VMD. Run cross-mode Granger causality:
    y_var_IMF(j) Granger-causes x_btc_IMF(i).
    statsmodels expects pair columns: [dependent, causing] = [BTC_IMF, VAR_IMF]
    """
    # Align both series first
    subset = pd.concat([x_btc, y_var], axis=1).dropna()
    subset.columns = ["BTC", "VAR"]
    if len(subset) <= (max_gc_lag + 5):
        return

    # VMD decomposition (robust alignment)
    imfs_btc = vmd_to_df(subset["BTC"])
    imfs_var = vmd_to_df(subset["VAR"])

    if imfs_btc.empty or imfs_var.empty:
        return

    common_idx = imfs_btc.index.intersection(imfs_var.index)
    imfs_btc = imfs_btc.loc[common_idx]
    imfs_var = imfs_var.loc[common_idx]

    # Cross-IMF Granger
    for i in range(K):
        for j in range(K):
            pair = pd.concat([imfs_btc.iloc[:, i], imfs_var.iloc[:, j]], axis=1).dropna()
            if len(pair) > max_gc_lag + 2:
                pair.columns = ["BTC_IMF", "VAR_IMF"]
                # setting verbose=False to keep the console clean => actually triggering warnings
                test_result = grangercausalitytests(pair[["BTC_IMF", "VAR_IMF"]], maxlag=max_gc_lag)#, verbose=False)

                for lag in range(1, max_gc_lag + 1):
                    f_test = test_result[lag][0]["ssr_ftest"]  # (F, p, df_denom, df_num)
                    line_test_res = {
                        "BTC_IMF": i + 1,
                        "VAR_IMF": j + 1,
                        "VAR_NAME": y_var.name,
                        "F_stat": float(f_test[0]),
                        "p_value": float(f_test[1]),
                        "Lag": lag,
                        "Obs": int(pair.shape[0]),
                    }
                    print(line_test_res)
                    results.append(line_test_res)

In [5]:
# Loop over all variables
for var in stationary_predictors:
    print(f"\n===== {target} vs. {var} =====")
    series_y = df[var]
    subset = pd.concat([btc, series_y], axis=1).dropna()
    if subset.empty:
        continue
    series_x = subset[target]
    series_y = subset[var]
    vmd_granger(series_x, series_y, max_gc_lag=MAX_GC_LAG)

results_df = pd.DataFrame(results)
results_df


===== btc_ret vs. mp_exp_d =====

Granger Causality
number of lags (no zero) 1
ssr based F test:         F=0.2093  , p=0.6475  , df_denom=540, df_num=1
ssr based chi2 test:   chi2=0.2105  , p=0.6464  , df=1
likelihood ratio test: chi2=0.2104  , p=0.6464  , df=1
parameter F test:         F=0.2093  , p=0.6475  , df_denom=540, df_num=1

Granger Causality
number of lags (no zero) 2
ssr based F test:         F=0.3303  , p=0.7189  , df_denom=537, df_num=2
ssr based chi2 test:   chi2=0.6667  , p=0.7165  , df=2
likelihood ratio test: chi2=0.6663  , p=0.7167  , df=2
parameter F test:         F=0.3303  , p=0.7189  , df_denom=537, df_num=2

Granger Causality
number of lags (no zero) 3
ssr based F test:         F=0.2497  , p=0.8615  , df_denom=534, df_num=3
ssr based chi2 test:   chi2=0.7590  , p=0.8592  , df=3
likelihood ratio test: chi2=0.7585  , p=0.8594  , df=3
parameter F test:         F=0.2497  , p=0.8615  , df_denom=534, df_num=3

Granger Causality
number of lags (no zero) 4
ssr based F te

,BTC_IMF,VAR_IMF,VAR_NAME,F_stat,p_value,Lag,Obs
0,1,1,mp_exp_d,0.209299,0.647502,1,544
1,1,1,mp_exp_d,0.330278,0.718870,2,544
2,1,1,mp_exp_d,0.249731,0.861535,3,544
3,1,1,mp_exp_d,0.253606,0.907457,4,544
4,1,1,mp_exp_d,0.323131,0.899093,5,544
...,...,...,...,...,...,...,...
967,3,3,gt_climate,1.776871,0.170160,2,544
968,3,3,gt_climate,1.318589,0.267479,3,544
969,3,3,gt_climate,0.703717,0.589640,4,544
970,3,3,gt_climate,0.538115,0.747441,5,544


In [6]:

# 1. Define the mapping dictionary
names = {
    "btc_ret": "Btc", "mp_exp_d": "MPE", 
    "news_sent": "NewsSent", "policy_risk_gr": "PolUncert",
    "sp500_ret": "SP500", "brent_gr": "Brent", 
    "gold_ret": "Gold", "hy_ret": "HighYield",
    "gpr_gr": "GeopolRisk", "vix_gr": "VIX", "dxy_gr": "USDollar",
    "emv": "Infect", "claims_dlog": "JoblessClaim", "eer_dlog": "ExchRate", 
    "ffr_d2": "FFR", "infl5y_d": "5yInflExp",
    "gt_infl_d": "GgleInfl", "gt_recess": "GgleReces", "gt_climate": "GgleClimate",
}

# 2. Map the VAR_NAME column
results_df["VAR_NAME"] = results_df["VAR_NAME"].map(names).fillna(results_df["VAR_NAME"])

# 3. Create the formula generation function
def generate_formula(row):
    # .replace("_", "\\_") ensures names like "News_Sent" become "News\_Sent"
    var = str(row['VAR_NAME']).replace("_", "\\_")
    
    btc_idx = int(row['BTC_IMF'])
    var_idx = int(row['VAR_IMF'])
    lag_val = int(row['Lag'])
    
    # Using your updated requested format
    return fr"$\textit{{{var}}}({var_idx}, {lag_val})$"

# 4. Apply the formula to a new column
results_df['formula'] = results_df.apply(generate_formula, axis=1)

# 5. Filter for significance
filtered_df = results_df[results_df['p_value'].astype(float) < 0.05].copy()

filtered_df['F_stat'] = filtered_df['F_stat'].map('{:.3f}'.format)
filtered_df['p_value'] = filtered_df['p_value'].map('{:.3f}'.format)

# 6. Split into three DataFrames based on BTC_IMF
df1 = filtered_df[filtered_df['BTC_IMF'] == 1][["formula", "F_stat", "p_value"]].reset_index(drop=True)
df2 = filtered_df[filtered_df['BTC_IMF'] == 2][["formula", "F_stat", "p_value"]].reset_index(drop=True)
df3 = filtered_df[filtered_df['BTC_IMF'] == 3][["formula", "F_stat", "p_value"]].reset_index(drop=True)

# Rename columns so they don't overlap (Optional, but helps in identification)
df1.columns = ['Formula (IMF1)', 'F-stat', 'p-val']
df2.columns = ['Formula (IMF2)', 'F-stat', 'p-val']
df3.columns = ['Formula (IMF3)', 'F-stat', 'p-val']

# 7. Concatenate them horizontally
# This puts IMF1, IMF2, and IMF3 side-by-side
final_table = pd.concat([df1, df2, df3], axis=1).fillna("")

display(final_table)

# 8. Generate the LaTeX output
# We use a very wide column format to accommodate 9 columns total
latex_output = final_table.to_latex(
    index=False, 
    escape=False, 
    column_format='lrr|lrr|lrr', # Three groups of (Formula, F, p)
    caption='Comparison of IMF Results for Bitcoin 1, 2, and 3',
    label='tab:comparison_results'
)

print(latex_output)

,Formula (IMF1),F-stat,p-val,Formula (IMF2),F-stat,p-val,Formula (IMF3),F-stat,p-val
0,"$\textit{NewsSent}(1, 2)$",3.842,0.022,"$\textit{SP500}(1, 2)$",4.439,0.012,"$\textit{MPE}(2, 2)$",6.495,0.002
1,"$\textit{NewsSent}(3, 1)$",11.226,0.001,"$\textit{SP500}(1, 4)$",5.267,0.000,"$\textit{MPE}(2, 3)$",3.991,0.008
2,"$\textit{NewsSent}(3, 2)$",4.978,0.007,"$\textit{SP500}(1, 5)$",5.428,0.000,"$\textit{SP500}(1, 4)$",8.199,0.000
3,"$\textit{SP500}(1, 1)$",9.381,0.002,"$\textit{SP500}(1, 6)$",4.735,0.000,"$\textit{SP500}(1, 5)$",7.392,0.000
4,"$\textit{SP500}(1, 2)$",26.260,0.000,"$\textit{SP500}(2, 1)$",4.949,0.027,"$\textit{SP500}(1, 6)$",7.020,0.000
5,"$\textit{SP500}(1, 3)$",5.328,0.001,"$\textit{SP500}(2, 2)$",10.917,0.000,"$\textit{SP500}(3, 1)$",9.541,0.002
6,"$\textit{SP500}(1, 4)$",4.134,0.003,"$\textit{SP500}(2, 3)$",7.513,0.000,"$\textit{Brent}(3, 1)$",7.823,0.005
7,"$\textit{SP500}(1, 5)$",3.351,0.005,"$\textit{SP500}(2, 4)$",3.451,0.008,"$\textit{Brent}(3, 2)$",3.215,0.041
8,"$\textit{SP500}(1, 6)$",2.984,0.007,"$\textit{SP500}(2, 5)$",2.859,0.015,"$\textit{Brent}(3, 3)$",4.152,0.006
9,"$\textit{SP500}(3, 4)$",2.619,0.034,"$\textit{Gold}(1, 4)$",3.254,0.012,"$\textit{Gold}(3, 1)$",12.000,0.001


\begin{table}
\caption{Comparison of IMF Results for Bitcoin 1, 2, and 3}
\label{tab:comparison_results}
\begin{tabular}{lrr|lrr|lrr}
\toprule
Formula (IMF1) & F-stat & p-val & Formula (IMF2) & F-stat & p-val & Formula (IMF3) & F-stat & p-val \\
\midrule
$\textit{NewsSent}(1, 2)$ & 3.842 & 0.022 & $\textit{SP500}(1, 2)$ & 4.439 & 0.012 & $\textit{MPE}(2, 2)$ & 6.495 & 0.002 \\
$\textit{NewsSent}(3, 1)$ & 11.226 & 0.001 & $\textit{SP500}(1, 4)$ & 5.267 & 0.000 & $\textit{MPE}(2, 3)$ & 3.991 & 0.008 \\
$\textit{NewsSent}(3, 2)$ & 4.978 & 0.007 & $\textit{SP500}(1, 5)$ & 5.428 & 0.000 & $\textit{SP500}(1, 4)$ & 8.199 & 0.000 \\
$\textit{SP500}(1, 1)$ & 9.381 & 0.002 & $\textit{SP500}(1, 6)$ & 4.735 & 0.000 & $\textit{SP500}(1, 5)$ & 7.392 & 0.000 \\
$\textit{SP500}(1, 2)$ & 26.260 & 0.000 & $\textit{SP500}(2, 1)$ & 4.949 & 0.027 & $\textit{SP500}(1, 6)$ & 7.020 & 0.000 \\
$\textit{SP500}(1, 3)$ & 5.328 & 0.001 & $\textit{SP500}(2, 2)$ & 10.917 & 0.000 & $\textit{SP500}(3, 1)$ & 9.541 & 0.